# InsPLAD Final Gold QA — Multi-model Baseline

최종 QA(`gold_v1`)를 고정한 뒤 image-only baseline을 처음부터 다시 평가합니다.

모델:
- Qwen3.5-4B
- InternVL3.5-8B
- MiniCPM-V-4.5
- LLaVA-OneVision-7B
- Luxia Document AI + Luxia Chat

평가:
- Q1: Anomaly F1 / Accuracy
- Q2: Defect Macro-F1 / Accuracy
- Q3: Localization Accuracy (Q3 GT가 있는 subset만)
- Overall MCQA Accuracy / Valid Output Rate

결과는 `MyDrive/lvlm_insplad_results/gold_baseline_v1/`에 저장합니다.
중간 prediction은 5문항마다 저장되므로 Colab이 끊겨도 같은 셀을 다시 실행하면 재사용합니다.


In [ ]:
# 0) 설정
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"

QA_FOLDER = "lvlm_insplad_qa/gold_v1"
RESULT_FOLDER = "lvlm_insplad_results/gold_baseline_v1"
DRIVE_DATA_DIR = None  # 예: "datasets/InsPLAD"

MODELS_TO_RUN = [
    "qwen3_5_4b",
    "internvl3_5_8b",
    "minicpm_v_4_5",
    "llava_onevision_7b",
    "luxia",
]

CHECKPOINT_EVERY = 5
SUP_ZIP_NAME = "supervised_fault_classification.zip"
EXPECTED_SUP_ZIP_SIZE = 874_753_167


In [ ]:
# 1) Drive mount + repo clone + 의존성 설치
import base64, json, os, shutil, subprocess, sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
RESULT_DIR = DRIVE_ROOT / RESULT_FOLDER
RESULT_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, cwd=None):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:7]) + (" ..." if len(cmd) > 7 else ""))
    subprocess.run(cmd, cwd=cwd, check=True)

github_token = None
try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

clone = ["git"]
if github_token:
    auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    clone += ["-c", f"http.extraHeader=AUTHORIZATION: basic {auth}"]
clone += [
    "clone", "--depth", "1", "--branch", REPO_REF,
    f"https://github.com/{REPO}.git", str(REPO_DIR)
]
run(clone)

run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==5.14.0",
    "accelerate==1.14.0",
    "sentencepiece==0.2.2",
    "pillow",
    "pandas",
    "tqdm",
])

print("repo:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip())
print("QA:", QA_DIR)
print("results:", RESULT_DIR)


In [ ]:
# 2) Secrets + GPU 확인
import torch

for secret_name in ["HF_TOKEN", "LUXIA_API_KEY"]:
    value = None
    try:
        value = userdata.get(secret_name)
    except Exception:
        pass
    if value:
        os.environ[secret_name] = value
        print("PASS secret:", secret_name)
    else:
        print("WARN missing secret:", secret_name)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(props.total_memory / (1024**3), 1))
else:
    print("GPU 없음: HF 4개 모델은 실패 처리되고 Luxia만 실행 가능합니다.")

print("Luxia는 API 호출이 많으므로 LUXIA_API_KEY가 반드시 필요합니다.")


In [ ]:
# 3) 최종 QA와 source ZIP 확인
def find_unique_file(filename):
    roots = []
    if DRIVE_DATA_DIR:
        roots.append(DRIVE_ROOT / DRIVE_DATA_DIR)
    roots.append(DRIVE_ROOT)

    matches = []
    for root in roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.is_file():
            matches.append(direct)
        matches.extend(p for p in root.rglob(filename) if p.is_file())

    unique = []
    seen = set()
    for p in matches:
        key = str(p.resolve())
        if key not in seen:
            unique.append(p)
            seen.add(key)

    if not unique:
        raise FileNotFoundError(f"{filename} 을 Drive에서 찾지 못했습니다.")
    if len(unique) > 1:
        print("[주의] 여러 후보 중 첫 파일 사용:", unique[0])
    return unique[0]

if not QA_DIR.is_dir():
    raise FileNotFoundError(
        f"QA folder 없음: {QA_DIR}\n"
        "먼저 build_explainable_qa_gold_colab.ipynb를 완료하세요."
    )

master_files = sorted(QA_DIR.glob("master_qa_*.jsonl"))
loc_files = sorted(QA_DIR.glob("localization_qa_*.jsonl"))
if not master_files:
    raise FileNotFoundError("master_qa_*.jsonl 없음")

SUP_ZIP = find_unique_file(SUP_ZIP_NAME)
if SUP_ZIP.stat().st_size != EXPECTED_SUP_ZIP_SIZE:
    raise RuntimeError(
        f"supervised ZIP size mismatch: {SUP_ZIP.stat().st_size}"
    )

sys.path.insert(0, str(REPO_DIR / "scripts"))
import run_gold_baselines as gold_runner

rows = gold_runner.load_gold_qa(QA_DIR)
q_counts = {}
for row in rows:
    q_counts[row["question_type"]] = q_counts.get(row["question_type"], 0) + 1

print("SUP_ZIP:", SUP_ZIP)
print("QA counts:", q_counts)
print("총 questions:", len(rows))
print("QA SHA256:", gold_runner.qa_identity(rows))


In [ ]:
# 4) 5개 baseline 실행 — 끊기면 이 셀만 다시 실행
cmd = [
    sys.executable,
    REPO_DIR / "scripts/run_gold_baselines.py",
    "--qa-dir", QA_DIR,
    "--sup-zip", SUP_ZIP,
    "--out-dir", RESULT_DIR,
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--models", *MODELS_TO_RUN,
]
run(cmd)


In [ ]:
# 5) 결과 표
import pandas as pd

leaderboard_path = RESULT_DIR / "leaderboard.csv"
status_path = RESULT_DIR / "run_status.json"
asset_path = RESULT_DIR / "metrics_by_asset.csv"

if status_path.is_file():
    print(json.dumps(
        json.loads(status_path.read_text(encoding="utf-8")),
        ensure_ascii=False,
        indent=2
    ))

if leaderboard_path.is_file():
    leaderboard = pd.read_csv(leaderboard_path)
    display(
        leaderboard.sort_values(
            ["anomaly_f1", "defect_macro_f1", "localization_accuracy"],
            ascending=False
        )
    )
else:
    print("leaderboard.csv 없음")

if asset_path.is_file():
    by_asset = pd.read_csv(asset_path)
    display(by_asset)


In [ ]:
# 6) Prediction review CSV 생성
import pandas as pd

qa_rows = gold_runner.read_jsonl(RESULT_DIR / "run_qa.jsonl")
qa_by_id = {row["id"]: row for row in qa_rows}

review_rows = []
for model_key in MODELS_TO_RUN:
    pred_path = RESULT_DIR / "predictions" / f"{model_key}.jsonl"
    if not pred_path.is_file():
        continue
    for pred in gold_runner.read_jsonl(pred_path):
        qa = qa_by_id[pred["id"]]
        review_rows.append({
            "model_key": model_key,
            "id": qa["id"],
            "core_id": qa["core_id"],
            "asset_category": qa["asset_category"],
            "question_type": qa["question_type"],
            "gt_choice": qa["answer"],
            "predicted_choice": pred.get("predicted_choice"),
            "correct": pred.get("predicted_choice") == qa["answer"],
            "anomaly_gt": qa["anomaly_gt"],
            "defect_gt": qa["defect_gt"],
            "question": qa["question"],
            "answer_text": qa.get("answer_text", ""),
            "raw_output": pred.get("output", ""),
            "error": pred.get("error"),
            "latency_s": pred.get("latency_s"),
        })

review = pd.DataFrame(review_rows)
review_path = RESULT_DIR / "prediction_review.csv"
review.to_csv(review_path, index=False)
print("saved:", review_path)
print("rows:", len(review))
if len(review):
    display(review[review["correct"] == False].head(50))


## 다음 단계

이 baseline 결과를 고정한 뒤 성능 개선 실험으로 넘어갑니다.

1. Image-only baseline ← 이 노트북
2. Normal Reference
3. Vision Expert → ROI / anomaly map
4. Reference + Vision Expert 조합
5. Agent 구조
6. 동일 QA에서 ablation 비교

baseline이 끝난 후 `leaderboard.csv`, `prediction_review.csv`, `run_status.json`을 ChatGPT에 주면 실패 유형부터 분석하면 됩니다.
